# Hal-13k Trajectory vs Camera Consistency Report

This notebook scans every episode under `aeroduo/data/Hal-13k` and compares:

- `high_uav_traj.json['rel_state']` against the number of images in `bevcamera/`
- `low_uav_traj.json['rel_state']` against the number of images in `frontcamera/`

For each trajectory file, the notebook reports both the raw trajectory length (`len(rel_state)`) and the number of unique trajectory states.

It also computes **explicit aligned-pair statistics** so the distribution is easy to see:

- **Low-UAV/frontcamera aligned pair count**: the number of pairs in episodes where `low_uav_traj['rel_state']` exactly matches `frontcamera` frame count.
- **High-UAV/bevcamera aligned pair count**: the number of `bevcamera` frame indices whose filename stems map to valid indices inside `high_uav_traj['rel_state']`.

Finally, the notebook flags episodes where the trajectory length and camera frame count do not match, including a dedicated table for episodes where both high- and low-UAV checks fail.


In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    from IPython.display import display
except ImportError:
    display = print

IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg"}
DATASET_ROOT_CANDIDATES = [
    Path("../data/Hal-13k"),
    Path("aeroduo/data/Hal-13k"),
    Path("./data/Hal-13k"),
]

for candidate in DATASET_ROOT_CANDIDATES:
    resolved = candidate.resolve()
    if resolved.exists():
        DATASET_ROOT = resolved
        break
else:
    searched = ", ".join(str(path.resolve()) for path in DATASET_ROOT_CANDIDATES)
    raise FileNotFoundError(f"Could not locate Hal-13k dataset. Checked: {searched}")


def iter_episode_dirs(dataset_root: Path):
    for scene_dir in sorted(
        path
        for path in dataset_root.iterdir()
        if path.is_dir() and path.name.startswith("Carla_")
    ):
        for episode_dir in sorted(path for path in scene_dir.iterdir() if path.is_dir()):
            yield scene_dir.name, episode_dir


def join_issues(*issues: str | None) -> str | None:
    merged = [issue for issue in issues if issue]
    return "; ".join(merged) if merged else None


def load_rel_state_stats(traj_path: Path) -> dict[str, Any]:
    if not traj_path.exists():
        return {
            "trajectory_exists": False,
            "trajectory_length": None,
            "unique_trajectory_length": None,
            "trajectory_issue": "missing trajectory file",
        }

    try:
        with traj_path.open("r", encoding="utf-8") as f:
            payload = json.load(f)
    except Exception as exc:
        return {
            "trajectory_exists": True,
            "trajectory_length": None,
            "unique_trajectory_length": None,
            "trajectory_issue": f"load error: {exc}",
        }

    rel_state = payload.get("rel_state")
    if not isinstance(rel_state, list):
        return {
            "trajectory_exists": True,
            "trajectory_length": None,
            "unique_trajectory_length": None,
            "trajectory_issue": "invalid or missing 'rel_state'",
        }

    unique_trajectory_length = len({json.dumps(row, sort_keys=True) for row in rel_state})
    return {
        "trajectory_exists": True,
        "trajectory_length": len(rel_state),
        "unique_trajectory_length": unique_trajectory_length,
        "trajectory_issue": None,
    }


def load_camera_stats(camera_dir: Path) -> dict[str, Any]:
    if not camera_dir.exists():
        return {
            "camera_exists": False,
            "camera_frame_count": 0,
            "camera_frame_indices": [],
            "camera_issue": "missing camera directory",
        }

    frame_paths = sorted(
        path
        for path in camera_dir.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    )
    frame_indices: list[int] = []
    for path in frame_paths:
        try:
            frame_indices.append(int(path.stem))
        except ValueError:
            continue

    return {
        "camera_exists": True,
        "camera_frame_count": len(frame_paths),
        "camera_frame_indices": frame_indices,
        "camera_issue": None,
    }


def compare_counts(
    trajectory_length: int | None,
    unique_trajectory_length: int | None,
    camera_frame_count: int,
) -> dict[str, Any]:
    return {
        "trajectory_length_matches_camera": (
            None if trajectory_length is None else trajectory_length == camera_frame_count
        ),
        "unique_trajectory_matches_camera": (
            None
            if unique_trajectory_length is None
            else unique_trajectory_length == camera_frame_count
        ),
        "trajectory_length_minus_camera": (
            None if trajectory_length is None else trajectory_length - camera_frame_count
        ),
        "unique_trajectory_minus_camera": (
            None
            if unique_trajectory_length is None
            else unique_trajectory_length - camera_frame_count
        ),
    }


def bool_count(rows: list[dict[str, Any]], key: str, value: bool) -> int:
    return sum(row[key] is value for row in rows)


def median(values: list[int | float]) -> float | None:
    if not values:
        return None
    sorted_values = sorted(values)
    mid = len(sorted_values) // 2
    if len(sorted_values) % 2 == 1:
        return float(sorted_values[mid])
    return float((sorted_values[mid - 1] + sorted_values[mid]) / 2)


def summarize_counts(values: list[int], pair_type: str) -> dict[str, Any]:
    if not values:
        return {
            "pair_type": pair_type,
            "episode_count": 0,
            "min": None,
            "max": None,
            "range": None,
            "mean": None,
            "median": None,
        }

    return {
        "pair_type": pair_type,
        "episode_count": len(values),
        "min": min(values),
        "max": max(values),
        "range": max(values) - min(values),
        "mean": float(sum(values) / len(values)),
        "median": median(values),
    }


def build_episode_row(scene: str, episode_dir: Path) -> dict[str, Any]:
    high_stats = load_rel_state_stats(episode_dir / "high_uav_traj.json")
    low_stats = load_rel_state_stats(episode_dir / "low_uav_traj.json")
    bev_stats = load_camera_stats(episode_dir / "bevcamera")
    front_stats = load_camera_stats(episode_dir / "frontcamera")

    high_compare = compare_counts(
        high_stats["trajectory_length"],
        high_stats["unique_trajectory_length"],
        bev_stats["camera_frame_count"],
    )
    low_compare = compare_counts(
        low_stats["trajectory_length"],
        low_stats["unique_trajectory_length"],
        front_stats["camera_frame_count"],
    )

    high_length_mismatch = high_compare["trajectory_length_matches_camera"] is False
    low_length_mismatch = low_compare["trajectory_length_matches_camera"] is False

    # high_uav_traj.json is now positionally indexed (one entry per BEV frame),
    # so aligned pairs == trajectory_length == bevcamera_frame_count.
    aligned_high_pair_count = None
    if high_stats["trajectory_length"] is not None:
        aligned_high_pair_count = high_stats["trajectory_length"]

    # Cross-UAV aligned pairs: BEV frame indices that also exist in frontcamera.
    # Used to determine how many graph nodes have a matching low-UAV observation.
    cross_aligned_pair_count = None
    if high_stats["trajectory_length"] is not None and bev_stats["camera_frame_indices"]:
        front_index_set = set(front_stats["camera_frame_indices"])
        cross_aligned_pair_count = sum(
            1 for idx in bev_stats["camera_frame_indices"] if idx in front_index_set
        )

    low_aligned_pair_count = None
    if low_compare["trajectory_length_matches_camera"] is True:
        low_aligned_pair_count = front_stats["camera_frame_count"]

    return {
        "scene": scene,
        "episode_id": episode_dir.name,
        "episode_path": str(episode_dir),
        "high_trajectory_length": high_stats["trajectory_length"],
        "high_unique_trajectory_length": high_stats["unique_trajectory_length"],
        "bevcamera_frame_count": bev_stats["camera_frame_count"],
        "high_length_matches_bevcamera": high_compare["trajectory_length_matches_camera"],
        "high_unique_matches_bevcamera": high_compare["unique_trajectory_matches_camera"],
        "high_length_minus_bevcamera": high_compare["trajectory_length_minus_camera"],
        "high_unique_minus_bevcamera": high_compare["unique_trajectory_minus_camera"],
        "high_issue": join_issues(high_stats["trajectory_issue"], bev_stats["camera_issue"]),
        "aligned_high_pair_count": aligned_high_pair_count,
        "cross_aligned_pair_count": cross_aligned_pair_count,
        "low_trajectory_length": low_stats["trajectory_length"],
        "low_unique_trajectory_length": low_stats["unique_trajectory_length"],
        "frontcamera_frame_count": front_stats["camera_frame_count"],
        "low_length_matches_frontcamera": low_compare["trajectory_length_matches_camera"],
        "low_unique_matches_frontcamera": low_compare["unique_trajectory_matches_camera"],
        "low_length_minus_frontcamera": low_compare["trajectory_length_minus_camera"],
        "low_unique_minus_frontcamera": low_compare["unique_trajectory_minus_camera"],
        "low_issue": join_issues(low_stats["trajectory_issue"], front_stats["camera_issue"]),
        "low_aligned_pair_count": low_aligned_pair_count,
        "high_length_mismatch": high_length_mismatch,
        "low_length_mismatch": low_length_mismatch,
        "both_length_mismatch": high_length_mismatch and low_length_mismatch,
        "any_flag": bool(
            high_length_mismatch
            or low_length_mismatch
            or join_issues(high_stats["trajectory_issue"], bev_stats["camera_issue"])
            or join_issues(low_stats["trajectory_issue"], front_stats["camera_issue"])
        ),
    }


In [2]:
rows = [build_episode_row(scene, episode_dir) for scene, episode_dir in iter_episode_dirs(DATASET_ROOT)]

aligned_low_pair_counts = [
    row["low_aligned_pair_count"]
    for row in rows
    if row["low_aligned_pair_count"] is not None
]
aligned_high_pair_counts = [
    row["aligned_high_pair_count"]
    for row in rows
    if row["aligned_high_pair_count"] is not None
]
cross_aligned_pair_counts = [
    row["cross_aligned_pair_count"]
    for row in rows
    if row["cross_aligned_pair_count"] is not None
]
camera_frame_count_differences = [
    row["bevcamera_frame_count"] - row["frontcamera_frame_count"]
    for row in rows
]

summary = {
    "dataset_root": str(DATASET_ROOT),
    "episodes": len(rows),
    "high_length_matches": bool_count(rows, "high_length_matches_bevcamera", True),
    "high_length_mismatches": bool_count(rows, "high_length_matches_bevcamera", False),
    "high_unique_matches": bool_count(rows, "high_unique_matches_bevcamera", True),
    "high_unique_mismatches": bool_count(rows, "high_unique_matches_bevcamera", False),
    "high_issues": sum(row["high_issue"] is not None for row in rows),
    "low_length_matches": bool_count(rows, "low_length_matches_frontcamera", True),
    "low_length_mismatches": bool_count(rows, "low_length_matches_frontcamera", False),
    "low_unique_matches": bool_count(rows, "low_unique_matches_frontcamera", True),
    "low_unique_mismatches": bool_count(rows, "low_unique_matches_frontcamera", False),
    "low_issues": sum(row["low_issue"] is not None for row in rows),
    "episodes_with_any_flag": sum(row["any_flag"] for row in rows),
    "episodes_with_both_length_mismatch": sum(row["both_length_mismatch"] for row in rows),
}

low_pair_stats = summarize_counts(
    aligned_low_pair_counts,
    "low_uav_traj <-> frontcamera aligned pairs",
)
high_pair_stats = summarize_counts(
    aligned_high_pair_counts,
    "high_uav_traj <-> bevcamera aligned pairs",
)
cross_pair_stats = summarize_counts(
    cross_aligned_pair_counts,
    "cross-UAV aligned pairs (BEV frame indices present in both bevcamera and frontcamera)",
)
camera_difference_stats = summarize_counts(
    camera_frame_count_differences,
    "bevcamera_count - frontcamera_count per episode",
)

high_flag_columns = [
    "scene",
    "episode_id",
    "high_trajectory_length",
    "high_unique_trajectory_length",
    "bevcamera_frame_count",
    "aligned_high_pair_count",
    "cross_aligned_pair_count",
    "high_length_matches_bevcamera",
    "high_unique_matches_bevcamera",
    "high_length_minus_bevcamera",
    "high_unique_minus_bevcamera",
    "high_issue",
    "episode_path",
]
low_flag_columns = [
    "scene",
    "episode_id",
    "low_trajectory_length",
    "low_unique_trajectory_length",
    "frontcamera_frame_count",
    "low_aligned_pair_count",
    "low_length_matches_frontcamera",
    "low_unique_matches_frontcamera",
    "low_length_minus_frontcamera",
    "low_unique_minus_frontcamera",
    "low_issue",
    "episode_path",
]
both_flag_columns = [
    "scene",
    "episode_id",
    "high_trajectory_length",
    "bevcamera_frame_count",
    "aligned_high_pair_count",
    "cross_aligned_pair_count",
    "high_length_minus_bevcamera",
    "low_trajectory_length",
    "frontcamera_frame_count",
    "low_aligned_pair_count",
    "low_length_minus_frontcamera",
    "episode_path",
]

if pd is not None:
    report_df = pd.DataFrame(rows).sort_values(["scene", "episode_id"]).reset_index(drop=True)
    summary_df = pd.DataFrame.from_dict(summary, orient="index", columns=["value"])
    low_pair_stats_df = pd.DataFrame.from_dict(
        {key: value for key, value in low_pair_stats.items() if key != "pair_type"},
        orient="index",
        columns=["value"],
    )
    high_pair_stats_df = pd.DataFrame.from_dict(
        {key: value for key, value in high_pair_stats.items() if key != "pair_type"},
        orient="index",
        columns=["value"],
    )
    cross_pair_stats_df = pd.DataFrame.from_dict(
        {key: value for key, value in cross_pair_stats.items() if key != "pair_type"},
        orient="index",
        columns=["value"],
    )
    camera_difference_stats_df = pd.DataFrame.from_dict(
        {key: value for key, value in camera_difference_stats.items() if key != "pair_type"},
        orient="index",
        columns=["value"],
    )

    high_flagged_df = report_df.loc[
        report_df["high_length_mismatch"] | report_df["high_issue"].notna(),
        high_flag_columns,
    ].reset_index(drop=True)
    low_flagged_df = report_df.loc[
        report_df["low_length_mismatch"] | report_df["low_issue"].notna(),
        low_flag_columns,
    ].reset_index(drop=True)
    both_flagged_df = report_df.loc[
        report_df["both_length_mismatch"],
        both_flag_columns,
    ].reset_index(drop=True)

    display(summary_df)
    print("\nLow-UAV/frontcamera aligned pair stats:")
    display(low_pair_stats_df)
    print("\nHigh-UAV/bevcamera aligned pair stats (one entry per BEV frame after subsampling):")
    display(high_pair_stats_df)
    print("\nCross-UAV aligned pair stats (BEV frame indices shared with frontcamera):")
    display(cross_pair_stats_df)
    print("\nPer-episode camera count difference stats (bevcamera - frontcamera):")
    display(camera_difference_stats_df)
    print("\nHigh-UAV episodes flagged (length mismatch or missing/invalid data):")
    display(high_flagged_df)
    print("\nLow-UAV episodes flagged (length mismatch or missing/invalid data):")
    display(low_flagged_df)
    print("\nEpisodes where BOTH high- and low-UAV trajectory lengths differ from camera counts:")
    display(both_flagged_df)
    print("\n`report_df` contains the full per-episode report.")
else:
    from pprint import pprint

    high_flagged_rows = [
        {key: row[key] for key in high_flag_columns}
        for row in rows
        if row["high_length_mismatch"] or row["high_issue"] is not None
    ]
    low_flagged_rows = [
        {key: row[key] for key in low_flag_columns}
        for row in rows
        if row["low_length_mismatch"] or row["low_issue"] is not None
    ]
    both_flagged_rows = [
        {key: row[key] for key in both_flag_columns}
        for row in rows
        if row["both_length_mismatch"]
    ]

    pprint(summary)
    print("\nLow-UAV/frontcamera aligned pair stats:")
    pprint(low_pair_stats)
    print("\nHigh-UAV/bevcamera aligned pair stats (one entry per BEV frame after subsampling):")
    pprint(high_pair_stats)
    print("\nCross-UAV aligned pair stats (BEV frame indices shared with frontcamera):")
    pprint(cross_pair_stats)
    print("\nPer-episode camera count difference stats (bevcamera - frontcamera):")
    pprint(camera_difference_stats)
    print("\nHigh-UAV flagged episodes:")
    pprint(high_flagged_rows)
    print("\nLow-UAV flagged episodes:")
    pprint(low_flagged_rows)
    print("\nEpisodes where both trajectory lengths mismatch camera counts:")
    pprint(both_flagged_rows)


,value
dataset_root,/storage/project/r-cj124-0/sibidapo3/8750/aero...
episodes,5377
high_length_matches,5377
high_length_mismatches,0
high_unique_matches,5377
high_unique_mismatches,0
high_issues,0
low_length_matches,5377
low_length_mismatches,0
low_unique_matches,5377



Low-UAV/frontcamera aligned pair stats:


,value
episode_count,5377.000000
min,5.000000
max,193.000000
range,188.000000
mean,45.649247
median,43.000000



High-UAV/bevcamera aligned pair stats (one entry per BEV frame after subsampling):


,value
episode_count,5377.000000
min,2.000000
max,193.000000
range,191.000000
mean,40.349265
median,38.000000


...


Low-UAV episodes flagged (length mismatch or missing/invalid data):


,scene,episode_id,low_trajectory_length,low_unique_trajectory_length,frontcamera_frame_count,low_aligned_pair_count,low_length_matches_frontcamera,low_unique_matches_frontcamera,low_length_minus_frontcamera,low_unique_minus_frontcamera,low_issue,episode_path



Episodes where BOTH high- and low-UAV trajectory lengths differ from camera counts:


,scene,episode_id,high_trajectory_length,bevcamera_frame_count,aligned_high_pair_count,cross_aligned_pair_count,high_length_minus_bevcamera,low_trajectory_length,frontcamera_frame_count,low_aligned_pair_count,low_length_minus_frontcamera,episode_path



`report_df` contains the full per-episode report.


In [3]:
bev_gt_front_rows = [
    row
    for row in rows
    if row["bevcamera_frame_count"] > row["frontcamera_frame_count"]
]

print(
    "Episodes where high_uav bevcamera frame count is greater than low_uav frontcamera frame count:",
    len(bev_gt_front_rows),
)

if pd is not None:
    bev_gt_front_df = (
        pd.DataFrame(bev_gt_front_rows)[
            [
                "scene",
                "episode_id",
                "bevcamera_frame_count",
                "frontcamera_frame_count",
                "episode_path",
            ]
        ]
        .assign(
            frame_count_delta=lambda df: (
                df["bevcamera_frame_count"] - df["frontcamera_frame_count"]
            )
        )
        .sort_values(["frame_count_delta", "scene", "episode_id"], ascending=[False, True, True])
        .reset_index(drop=True)
    )
    display(bev_gt_front_df)
else:
    from pprint import pprint

    pprint(
        [
            {
                "scene": row["scene"],
                "episode_id": row["episode_id"],
                "bevcamera_frame_count": row["bevcamera_frame_count"],
                "frontcamera_frame_count": row["frontcamera_frame_count"],
                "frame_count_delta": row["bevcamera_frame_count"] - row["frontcamera_frame_count"],
                "episode_path": row["episode_path"],
            }
            for row in bev_gt_front_rows
        ]
    )


Episodes where high_uav bevcamera frame count is greater than low_uav frontcamera frame count: 18


,scene,episode_id,bevcamera_frame_count,frontcamera_frame_count,episode_path,frame_count_delta
0,Carla_Town02,37cd0ea5-025f-4cd4-b11f-6217e4d5c15a,55,53,/storage/project/r-cj124-0/sibidapo3/8750/aero...,2
1,Carla_Town02,72fef8e7-5dd5-4eed-aaa5-a57185f6772c,71,69,/storage/project/r-cj124-0/sibidapo3/8750/aero...,2
2,Carla_Town02,05a09b7c-ff81-4009-9e25-ceb09c334766,19,18,/storage/project/r-cj124-0/sibidapo3/8750/aero...,1
3,Carla_Town02,0d9b0d02-cd89-4ce4-8004-b4b83a719b04,36,35,/storage/project/r-cj124-0/sibidapo3/8750/aero...,1
4,Carla_Town02,157a4fcd-0505-4350-9176-29745121865e,49,48,/storage/project/r-cj124-0/sibidapo3/8750/aero...,1
5,Carla_Town02,1740626f-80eb-49a0-985f-46196eab8956,64,63,/storage/project/r-cj124-0/sibidapo3/8750/aero...,1
6,Carla_Town02,1d8740a5-2150-4594-a3b2-455f30be9f04,25,24,/storage/project/r-cj124-0/sibidapo3/8750/aero...,1
7,Carla_Town02,1f3618f9-a8a5-4190-8f14-4c6107b01589,48,47,/storage/project/r-cj124-0/sibidapo3/8750/aero...,1
8,Carla_Town02,34683a5c-fedd-465a-a152-ff8535554f4d,43,42,/storage/project/r-cj124-0/sibidapo3/8750/aero...,1
9,Carla_Town02,37d08f4a-1992-4afb-9ad2-dd115f6ebaf3,38,37,/storage/project/r-cj124-0/sibidapo3/8750/aero...,1
